In [129]:
import pandas as pd
import numpy as np


In [130]:
from pathlib import Path

p = Path('data/processed/train_data.parquet')

if p.exists():
    df = pd.read_parquet(p)
else:
    alternatives = [
        Path('data/processed/train.parquet'),
        Path('data/train_data.parquet'),
        Path('../data/processed/train_data.parquet'),
    ]
    found = next((a for a in alternatives if a.exists()), None)
    if found:
        if found.suffix == '.parquet':
            df = pd.read_parquet(found)
            
    else:
        proc = list(Path('data/processed').iterdir()) if Path('data/processed').exists() else []
        data_root = list(Path('data').iterdir()) if Path('data').exists() else []
        raise FileNotFoundError(
            f"train_data.parquet not found at {p!s}. Checked alternatives: {[str(a) for a in alternatives]}. "
            f"Files in data/processed: {[str(x) for x in proc]}. Files in data: {[str(x) for x in data_root]}."
        )

In [131]:
df.head()

,user_id,measurement_date,hours_streaming,hours_social,hours_messaging,hours_gaming,is_peak_hour_user,is_weekend,age_group,plan_type,...,messaging_data_gb,gaming_data_gb,total_data_gb,total_usage_gb,top_activity,day_of_week,hour,churn_risk_score,arpu_zar,arpu_per_gb
0,SUB4895090,2026-03-27,1.27,6.00,0.15,0.00,1,0,35-44,Prepaid_Daily,...,0.00208,0.00000,4.45799,4.12503,streaming,Friday,0,0.741147,522.54,117.214260
1,SUB9369689,2026-03-30,0.33,0.22,1.79,0.31,0,0,25-34,Postpaid_Basic,...,0.02319,0.01750,0.24945,0.20206,streaming,Monday,0,0.996707,95.93,384.566045
2,SUB2099577,2026-04-11,0.54,1.57,0.39,0.10,0,0,25-34,Postpaid_Unlimited,...,0.00540,0.01295,0.61941,0.58525,streaming,Saturday,0,0.691823,79.00,127.540724
3,SUB5147036,2026-04-12,0.24,1.31,1.03,0.65,0,1,18-24,Postpaid_Premium,...,0.01589,0.08245,0.40725,0.41418,streaming,Sunday,0,0.694624,79.00,193.984039
4,SUB6877845,2026-04-10,1.90,0.68,0.01,0.22,0,0,18-24,Postpaid_Basic,...,0.00012,0.00968,4.66772,4.51226,streaming,Friday,0,0.738378,395.70,84.773723


In [132]:

from sklearn.base import BaseEstimator, TransformerMixin

# 1. Drop columns
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.cols, errors='ignore')


# 2. Date feature extraction
class DateFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['day_of_week'] = X['measurement_date'].dt.dayofweek
        X['month'] = X['measurement_date'].dt.month
        return X.drop(columns=['measurement_date'])


# 3. Cyclical encoding
class CyclicalFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X['day_of_week_sin'] = np.sin(2 * np.pi * X['day_of_week'] / 7)
        X['day_of_week_cos'] = np.cos(2 * np.pi * X['day_of_week'] / 7)

        X['month_sin'] = np.sin(2 * np.pi * X['month'] / 12)
        X['month_cos'] = np.cos(2 * np.pi * X['month'] / 12)

        return X.drop(columns=['day_of_week', 'month'])

In [133]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

from src.preprocessing import DropColumns, DateFeatures, CyclicalFeatures


def build_pipeline():

    #  Drop leakage + unnecessary columns
    remove_cols = [
        'user_id',
        'data_usage_category',
        'total_usage_gb',
        'streaming_data_gb',
        'social_data_gb',
        'messaging_data_gb',
        'gaming_data_gb',
        'top_activity',
        'day_of_week',
        'hour',
        'arpu_zar',
        'arpu_per_gb',
        'churn_risk_score'
    ]


    # Ordinal features

    ordinal_cols = ['age_group', 'plan_type', 'network_type']

    age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
    plan_order = [
        'Prepaid_Daily', 'Prepaid_Monthly',
        'Postpaid_Basic', 'Postpaid_Premium',
        'Postpaid_Unlimited'
    ]
    network_order = ['3G', '4G', '4G+', '5G']

    ordinal_encoder = OrdinalEncoder(
        categories=[age_order, plan_order, network_order]
    )


    # Nominal features

    nominal_cols = ['device_type']

    onehot = OneHotEncoder(
        drop='first',
        handle_unknown='ignore'
    )


    # Numeric features 

    numeric_cols = [
        'hours_streaming',
        'hours_social',
        'hours_messaging',
        'hours_gaming',
        'is_peak_hour_user',
        'is_weekend'
    ]


    # Column Transformer 

    preprocessor = ColumnTransformer(
        transformers=[
            ('ord', ordinal_encoder, ordinal_cols),
            ('nom', onehot, nominal_cols),
            ('num', 'passthrough', numeric_cols)
        ],
        remainder='drop'  #  prevents leakage
    )


    # Full Pipeline

    pipeline = Pipeline(steps=[
        ('drop_cols', DropColumns(remove_cols)),
        ('date_features', DateFeatures()),
        ('cyclical', CyclicalFeatures()),
        ('encoding', preprocessor)
    ])

    return pipeline

In [134]:
# Build pipeline
pipeline = build_pipeline()

# Save to the correct location
project_root = os.path.dirname(os.getcwd())  # Go up one level from notebooks
models_dir = os.path.join(project_root, 'models')

# Create models directory if it doesn't exist
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

models_path = os.path.join(models_dir, 'preprocessing_pipeline.pkl')

# Save pipeline
joblib.dump(pipeline, models_path)
print(f" Pipeline saved to: {models_path}")

 Pipeline saved to: g:\Study\DATA SCINCE\PROJECTS\POTFOLIO\telecom-consumption-intelligence\models\preprocessing_pipeline.pkl


In [ ]:
# Load the pipeline 

import joblib

project_root = os.path.dirname(os.getcwd())
models_path = os.path.join(project_root, 'models', 'preprocessing_pipeline.pkl')

preprocessing_pipeline = joblib.load(models_path)

        


In [136]:
from pathlib import Path

train_path = next(
    (
        p for p in [
            Path('../data/processed/train_data.parquet'),
            Path('data/processed/train_data.parquet'),
            Path('data/train_data.parquet'),
        ]
        if p.exists()
    ),
    Path('../data/processed/train_data.parquet')
)

df = pd.read_parquet(train_path)
df.head()

,user_id,measurement_date,hours_streaming,hours_social,hours_messaging,hours_gaming,is_peak_hour_user,is_weekend,age_group,plan_type,...,messaging_data_gb,gaming_data_gb,total_data_gb,total_usage_gb,top_activity,day_of_week,hour,churn_risk_score,arpu_zar,arpu_per_gb
0,SUB4895090,2026-03-27,1.27,6.00,0.15,0.00,1,0,35-44,Prepaid_Daily,...,0.00208,0.00000,4.45799,4.12503,streaming,Friday,0,0.741147,522.54,117.214260
1,SUB9369689,2026-03-30,0.33,0.22,1.79,0.31,0,0,25-34,Postpaid_Basic,...,0.02319,0.01750,0.24945,0.20206,streaming,Monday,0,0.996707,95.93,384.566045
2,SUB2099577,2026-04-11,0.54,1.57,0.39,0.10,0,0,25-34,Postpaid_Unlimited,...,0.00540,0.01295,0.61941,0.58525,streaming,Saturday,0,0.691823,79.00,127.540724
3,SUB5147036,2026-04-12,0.24,1.31,1.03,0.65,0,1,18-24,Postpaid_Premium,...,0.01589,0.08245,0.40725,0.41418,streaming,Sunday,0,0.694624,79.00,193.984039
4,SUB6877845,2026-04-10,1.90,0.68,0.01,0.22,0,0,18-24,Postpaid_Basic,...,0.00012,0.00968,4.66772,4.51226,streaming,Friday,0,0.738378,395.70,84.773723


In [137]:
#load the train data
train_df = pd.read_parquet(train_path)

# Split the data into features and target
from sklearn.model_selection import train_test_split

X = train_df.drop([
    "total_data_gb",          # target
    "total_usage_gb",
    "arpu_zar"
], axis=1)

y = train_df["total_data_gb"]

# 1. Split FIRST
X_train, X_val, y_train, y_val = train_test_split(X, y)

# 2. Fit on TRAIN only
preprocessing_pipeline.fit(X_train)

# 3. Transform both
X_train_processed = preprocessing_pipeline.transform(X_train)
X_val_processed   = preprocessing_pipeline.transform(X_val)


In [138]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Train model
model = LinearRegression()

model.fit(X_train_processed, y_train)

# Predictions
y_pred = model.predict(X_val_processed)

# Evaluation metrics
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

# Results
print("Linear Regression Performance")
print("-----------------------------")
print(f"RMSE: {rmse:.3f}")
print(f"MAE : {mae:.3f}")
print(f"R²  : {r2:.3f}")

Linear Regression Performance
-----------------------------
RMSE: 1.753
MAE : 1.058
R²  : 0.827


In [139]:
# Predict on training data
y_train_pred = model.predict(X_train_processed)

# Training metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("Training Performance")
print("--------------------")
print(f"RMSE: {train_rmse:.3f}")
print(f"MAE : {train_mae:.3f}")
print(f"R²  : {train_r2:.3f}")

Training Performance
--------------------
RMSE: 1.802
MAE : 1.095
R²  : 0.818
